# 模型整体架构（宏观视角）

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│                    大语言模型（LLM）                          │
│                  基于 Transformer Decoder                     │
├─────────────────────────────────────────────────────────────┤
│  输入层 (Input Embedding)                                   │
│    ├── Token Embedding: 词 → 向量 [vocab_size, hidden_dim]  │
│    └── Position Embedding: 位置信息编码                        │
├─────────────────────────────────────────────────────────────┤
│  堆叠的 N 个 Decoder Layer (例如 GPT-3 有 96 层)              │
│    ├── Layer 1: Dense / MoE                                 │
│    ├── Layer 2: Dense / MoE                                 │
│    ├── ...                                                  │
│    └── Layer N: Dense / MoE                                 │
├─────────────────────────────────────────────────────────────┤
│  输出层 (Output Head)                                        │
│    └── Linear: [hidden_dim] → [vocab_size] 预测下一个词      │
└─────────────────────────────────────────────────────────────┘

# 单层内部结构（微观视角）
##  标准 Dense Layer（非 MoE）

In [ ]:
┌─────────────────────────────────────────┐
│           Transformer Decoder Layer       │
├─────────────────────────────────────────┤
│  输入: x [batch, seq_len, hidden_dim]   │
├─────────────────────────────────────────┤
│  ┌─────────────────────────────────┐   │
│  │  1. Pre-Norm / Post-Norm        │   │
│  │     LayerNorm(x)                │   │
│  └─────────────────────────────────┘   │
│              ↓                         │
│  ┌─────────────────────────────────┐   │
│  │  2. Self-Attention (多头注意力)  │   │
│  │     • Q, K, V 投影 (3 个 Linear) │   │
│  │     • Scaled Dot-Product Attention│  │
│  │     • 输出投影 (1 个 Linear)     │   │
│  │     • 残差连接: x + Attention(x)  │   │
│  └─────────────────────────────────┘   │
│              ↓                         │
│  ┌─────────────────────────────────┐   │
│  │  3. Pre-Norm / Post-Norm        │   │
│  │     LayerNorm(x)                │   │
│  └─────────────────────────────────┘   │
│              ↓                         │
│  ┌─────────────────────────────────┐   │
│  │  4. FFN (Feed-Forward Network)  │   │
│  │     • Linear 1: [H] → [4H]      │   │
│  │     • 激活函数 (GELU/SwiGLU)    │   │
│  │     • Linear 2: [4H] → [H]      │   │
│  │     • 残差连接: x + FFN(x)      │   │
│  └─────────────────────────────────┘   │
│              ↓                         │
│  输出: x [batch, seq_len, hidden_dim]  │
└─────────────────────────────────────────┘

## MoE Layer（混合专家层）

In [ ]:
┌─────────────────────────────────────────────────────────────┐
│              MoE Transformer Layer                          │
├─────────────────────────────────────────────────────────────┤
│  输入: x [batch, seq_len, hidden_dim]                       │
├─────────────────────────────────────────────────────────────┤
│  1. Pre-Norm + Self-Attention（与 Dense 层完全相同）           │
├─────────────────────────────────────────────────────────────┤
│  2. Pre-Norm                                                  │
├─────────────────────────────────────────────────────────────┤
│  3. MoE Block（替换标准 FFN）                                 │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  3.1 Router（门控网络）                              │   │
│  │      Linear: [hidden_dim] → [num_experts]            │   │
│  │      Softmax → 每个 token 的 expert 权重分布         │   │
│  │      Top-k 选择（通常 k=1 或 2）                     │   │
│  │      输出: expert_indices [batch, seq_len, top_k]    │   │
│  │             expert_weights [batch, seq_len, top_k]   │   │
│  └─────────────────────────────────────────────────────┘   │
│                          ↓                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  3.2 Expert Network（专家网络）                       │   │
│  │                                                     │   │
│  │  ┌─────────┐  ┌─────────┐       ┌─────────┐      │   │
│  │  │ Expert 0│  │ Expert 1│  ...  │ Expert 7│      │   │
│  │  │  FFN-0  │  │  FFN-1  │       │  FFN-7  │      │   │
│  │  │ [H→4H→H]│  │ [H→4H→H]│       │ [H→4H→H]│      │   │
│  │  └─────────┘  └─────────┘       └─────────┘      │   │
│  │                                                     │   │
│  │  每个 Expert 结构 = 标准 FFN（但参数独立）            │   │
│  │  总参数量 = num_experts × expert_size               │   │
│  │  激活参数量 = top_k × expert_size（稀疏激活）       │   │
│  └─────────────────────────────────────────────────────┘   │
│                          ↓                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  3.3 加权聚合（Weighted Combine）                     │   │
│  │      对每个 token：                                   │   │
│  │      output = Σ(expert_weight_i × expert_i_output)    │   │
│  │      残差连接: x + MoE(x)                             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘

# 各层详细参数与计算
##  Self-Attention 层

In [ ]:
# 输入: x [batch, seq_len, hidden_dim=4096]

# 1. Q, K, V 投影
W_q = [hidden_dim, hidden_dim]      # 4096 × 4096
W_k = [hidden_dim, hidden_dim]      # 4096 × 4096  
W_v = [hidden_dim, hidden_dim]      # 4096 × 4096

# 2. 多头拆分: num_heads=32, head_dim=128
Q = x @ W_q  → [batch, seq_len, 32, 128]
K = x @ W_k  → [batch, seq_len, 32, 128]
V = x @ W_v  → [batch, seq_len, 32, 128]

# 3. Attention 计算
scores = Q @ K.T / sqrt(head_dim)   # [batch, 32, seq_len, seq_len]
attn = softmax(scores, dim=-1)
attn = dropout(attn)
out = attn @ V                      # [batch, 32, seq_len, 128]

# 4. 输出投影
W_o = [hidden_dim, hidden_dim]      # 4096 × 4096
output = out.transpose(1,2).reshape(batch, seq_len, -1) @ W_o

# Attention 总参数量: 4 × hidden_dim² = 4 × 4096² ≈ 67M

## 标准 FFN（Dense 层）

In [ ]:
# 输入: x [batch, seq_len, hidden_dim=4096]

# 方案 A: GELU FFN (GPT-3 风格)
W_1 = [hidden_dim, 4*hidden_dim]    # 4096 × 16384
W_2 = [4*hidden_dim, hidden_dim]    # 16384 × 4096

h = GELU(x @ W_1)                   # [batch, seq_len, 16384]
output = h @ W_2                    # [batch, seq_len, 4096]

# 参数量: 8 × hidden_dim² = 8 × 4096² ≈ 134M

# 方案 B: SwiGLU FFN (LLaMA/Mistral 风格，更优)
W_gate = [hidden_dim, 4*hidden_dim]  # 门控投影
W_up   = [hidden_dim, 4*hidden_dim]  # 上投影
W_down = [4*hidden_dim, hidden_dim]  # 下投影

gate = Swish(x @ W_gate)            # 门控信号
up   = x @ W_up                     # 上投影
h    = gate * up                    # 逐元素乘
output = h @ W_down

# 参数量: 12 × hidden_dim² ≈ 201M（但效果优于 GELU）

##  MoE 的 Expert FFN

In [ ]:
# 每个 Expert 结构 = 标准 FFN（但尺寸可能更小）
# 假设: num_experts=8, top_k=2, expert_ffn_dim=hidden_dim*2

Expert_i:
    W_gate_i = [hidden_dim, 2*hidden_dim]   # 4096 × 8192
    W_up_i   = [hidden_dim, 2*hidden_dim]   # 4096 × 8192
    W_down_i = [2*hidden_dim, hidden_dim]   # 8192 × 4096
    
# 单个 Expert 参数量: 3 × 4096 × 8192 ≈ 100M
# 8 个 Expert 总参数量: 8 × 100M = 800M
# 但激活参数量: top_k × 100M = 200M（稀疏！）

# Router（门控网络）
W_router = [hidden_dim, num_experts]  # 4096 × 8 ≈ 32K（可忽略）

router_logits = x @ W_router          # [batch, seq_len, 8]
weights = softmax(router_logits, dim=-1)
top_k_weights, top_k_indices = torch.topk(weights, k=2, dim=-1)

# 完整模型参数量对比
以 hidden_dim=4096, num_layers=32, vocab_size=32000 为例：

| 组件                 | Dense 模型            | MoE 模型 (8 experts, top-2)           |
| ------------------ | ------------------- | ----------------------------------- |
| Token Embedding    | 32000 × 4096 ≈ 131M | 131M                                |
| Position Embedding | 4096 × 4096 ≈ 16M   | 16M                                 |
| 32 层 Attention     | 32 × 67M ≈ 2.1B     | 2.1B                                |
| 32 层 FFN (Dense)   | 32 × 134M ≈ 4.3B    | —                                   |
| 32 层 MoE FFN       | —                   | 32 × 800M ≈ 25.6B                   |
| 32 层 Router        | —                   | 32 × 32K ≈ 1M                       |
| Output Head        | 4096 × 32000 ≈ 131M | 131M                                |
| **总参数量**           | **≈ 6.7B**          | **≈ 28B**                           |
| **激活参数量**          | **6.7B**            | **2.1B (Attn) + 6.4B (MoE) ≈ 8.5B** |

- MoE 的优势：总参数 28B（存储大），但前向激活只算 8.5B（计算快，显存省）。

# 现代 MoE 变体
##  细粒度 MoE（DeepSeek-MoE）

In [ ]:
标准 MoE: 8 experts, top-2 → 每个 expert 很大
细粒度: 64 experts, top-6 → 每个 expert 更小，更细粒度

DeepSeek-V2:
  - 64 experts + 2 shared experts（始终激活）
  - 6 个 routed experts
  - 每个 expert 的 ffn_dim = 2048（很小）
  - 总参数 236B，激活参数 21B

## Expert Choice（EC-MoE）

In [ ]:
标准: Token Choice → 每个 token 选 top-k expert
EC: Expert Choice → 每个 expert 选 top-k tokens（负载均衡更好）

优势:
  - 避免某些 expert 过载（"专家崩溃"）
  - 天然负载均衡，不需要辅助损失函数

## 共享 Expert（Shared Expert Isolation）

In [ ]:
┌─────────────────────────────────────────┐
│  MoE Layer                              │
│  ├── Shared Expert 0: 始终激活（通用知识）│
│  ├── Shared Expert 1: 始终激活           │
│  ├── Routed Expert 0-5: 动态选择（专业知识）│
│  └── Router: 选择 routed experts         │
└─────────────────────────────────────────┘

DeepSeek-V2/V3 使用此结构，shared expert 学通用模式，
routed experts 学专业领域知识。

# 层与层之间的数据流

In [ ]:
输入序列: "今天天气很好"

Token IDs: [101, 2345, 4567, 7890, 102]  # 假设的 token 编号
           ↓
Embedding: [5, 4096]  # batch=5 tokens, dim=4096
           ↓
┌────────────────────────────────────────┐
│ Layer 1 (Dense)                       │
│   x = LayerNorm(x)                     │
│   x = x + SelfAttention(x)            │
│   x = LayerNorm(x)                    │
│   x = x + FFN(x)                      │
│   输出: [5, 4096]                      │
└────────────────────────────────────────┘
           ↓
┌────────────────────────────────────────┐
│ Layer 2 (MoE)                         │
│   x = LayerNorm(x)                     │
│   x = x + SelfAttention(x)            │
│   x = LayerNorm(x)                    │
│   router = Router(x) → [5, 8]        │
│   top2_experts = [Expert3, Expert7]    │
│   expert_out = 0.7×Expert3(x) + 0.3×Expert7(x) │
│   x = x + expert_out                   │
│   输出: [5, 4096]                      │
└────────────────────────────────────────┘
           ↓
         ... 更多层 ...
           ↓
Output Head: [5, 4096] @ [4096, 32000] → [5, 32000]
           ↓
Softmax → 概率分布
           ↓
预测下一个词: "适合" (概率最高)

# 总结图
- 一句话：MoE 就是把 FFN 层复制 N 份，加上一个路由器，让不同 token 走不同的"专家"路径，总参数爆炸但计算量可控。

In [ ]:
大模型 = 嵌入层 + N × 解码层 + 输出头

解码层 = 注意力 + FFN/MoE + 残差 + LayerNorm

        ┌─────────────┐
        │  输入嵌入    │
        └──────┬──────┘
               ↓
        ┌─────────────┐
        │  Layer 1    │  Dense / MoE
        │  Attention  │
        │  FFN/MoE    │
        └──────┬──────┘
               ↓
              ...
               ↓
        ┌─────────────┐
        │  Layer N    │  Dense / MoE
        │  Attention  │
        │  FFN/MoE    │
        └──────┬──────┘
               ↓
        ┌─────────────┐
        │  输出头      │  Linear + Softmax
        └─────────────┘

MoE 的核心: 用 Router 动态选择 Expert，用稀疏激活换取大容量

# 两种设计：稀疏 MoE vs 密集 MoE

| 设计             | 哪层用 MoE      | 代表模型                              |
| -------------- | ------------ | --------------------------------- |
| **稀疏 MoE**（<font color='red'>常见</font>） | <font color='blue'>每隔几层替换一个 FFN </font>| Switch Transformer, GLaM, Mixtral |
| **密集 MoE**     | 几乎每层都是 MoE   | GPT-4 (推测), 部分实验模型                |
| **只在最后用**      | 输出前加 MoE     | 非主流，效果差                           |

## 主流做法：稀疏替换


In [ ]:
标准 Transformer (32层):
┌─────────────────────────────────────────┐
│ Layer 1:  Dense FFN                     │
│ Layer 2:  Dense FFN                     │
│ Layer 3:  MoE Layer  ← 每隔 1-3 层放一个 │
│ Layer 4:  Dense FFN                     │
│ Layer 5:  Dense FFN                     │
│ Layer 6:  MoE Layer                     │
│ ...                                     │
│ Layer 32: Dense FFN                     │
└─────────────────────────────────────────┘

典型比例：1/4 到 1/2 的层是 MoE

### 为什么不用每层？

| 原因    | 解释                     |
| ----- | ---------------------- |
| 训练稳定性 | 全 MoE 难以收敛，Router 学习困难 |
| 通信开销  | 每层 all-to-all 通信太频繁    |
| 负载均衡  | 层数越多，专家崩溃越严重           |
| 收益递减  | 前几层学通用特征，不需要太多专家       |


## 具体例子
### Mixtral 8x7B（Mistral AI）

In [ ]:
32 层，每 4 层中 1 层是 MoE？不，Mixtral 是：
- 所有 32 层都是 MoE！但 expert 很小

实际结构：
- 32 层，每层都是 MoE
- 8 个 expert，top-2
- 但 attention 共享，只有 FFN 部分是 MoE
- 总参数 46.7B，激活 12.9B

注意：Mixtral 是"全 MoE"的特例，训练很讲究技巧

### Switch Transformer (Google)

In [ ]:
12 层，每 1 层是 MoE（全 MoE）
- 1 expert / token (top-1)
- 专家数量 2048（极多）
- 用精度损失换取训练稳定

### GLaM (Google)

In [ ]:
64 层，每 4 层 1 个 MoE
- 64 层中 16 层是 MoE
- 64 个 expert
- 总参数 1.2T，激活 97B

### DeepSeek-V2

In [ ]:
60 层，全 MoE
- 64 routed experts + 2 shared experts
- 但用了 MLA（多头潜在注意力）压缩 KV cache
- 总参数 236B，激活 21B

## 为什么不在最后放 MoE？

In [ ]:
错误设计：
┌─────────────────────────────────────────┐
│ 32 层 Dense FFN                         │
│ ...                                     │
│ Layer 32: Dense FFN                     │
│                                         │
│ 最后加 MoE:                             │
│   Router → Expert 0-7 → Combine         │
│                                         │
│ 输出头                                  │
└─────────────────────────────────────────┘

问题：
1. 前 32 层都是"通用专家"，没有专业化
2. 最后 MoE 只能做"输出组合"，信息已固化
3. 中间层无法利用专家的专业化特征
4. 参数量浪费：前面 Dense 层已经很大

### <font color='red'> MoE 的价值在中间层：</font>
- 早期层：学词法、句法（通用，不需要专家）
- 中期层：学语义、知识（需要专家分化：数学/代码/文学）
- 后期层：学推理、生成（可以 MoE 也可以 Dense）


## 正确的层分布策略

In [ ]:
理想的 MoE 放置（基于层的功能）：

Layer 1-4:   Dense      ← 词嵌入、基本语法
Layer 5-8:   MoE        ← 知识开始分化
Layer 9-20:  MoE/Dense  ← 主要知识层，MoE 收益最大
Layer 21-28: MoE        ← 复杂推理
Layer 29-32: Dense      ← 输出格式化

实际工程中：
- 均匀间隔（最简单）：每 2-4 层放一个
- 前密后疏：前面 Dense 多，后面 MoE 多
- 按任务调：代码模型后面 MoE 多

## 总结

| 问题        | 答案                            |
| --------- | ----------------------------- |
| 每层都是 MoE？ | 可以，但难训练（Mixtral/DeepSeek 做到了） |
| 只在最后放？    | 不行，效果差，浪费参数                   |
| 主流做法？     | 每隔 1-4 层替换一个 FFN 为 MoE        |
| 最佳位置？     | 中间层（5-20层），<font color='red'>知识分化最明显的地方  </font>       |

- MoE 的核心思想是"让不同层负责不同抽象级别，同一层内让不同专家负责不同领域"，放在最后就失去了"分层学习"的意义。